# Fixed: ReAct-style agent example (modern LangChain)

This notebook replaces manual `ReActSingleInputOutputParser` usage with the modern `initialize_agent` helper and a simple tool set. It includes:

- LLM setup placeholder (replace with your LLM wrapper if needed)
- A simple calculator tool (safe demo)
- `initialize_agent` with `AgentType.ZERO_SHOT_REACT_DESCRIPTION`
- Example runs and notes about capturing intermediate steps

Make sure your environment has a compatible `langchain` version installed and your preferred LLM configured.

In [ ]:
# Imports - replace OpenAI with your preferred LLM wrapper if needed
from langchain import OpenAI
from langchain.agents import initialize_agent, Tool, AgentType
from langchain.schema import AgentAction, AgentFinish

# NOTE: If you use ChatOpenAI or another wrapper, import that instead:
# from langchain.chat_models import ChatOpenAI
# llm = ChatOpenAI(temperature=0)

# Create an LLM instance (placeholder). Replace with your configured LLM.
llm = OpenAI(temperature=0)  # adjust per your setup

# Simple tool: calculator (demo). In production, DO NOT use eval on untrusted input.
def simple_calc(expr: str) -> str:
    """Evaluate a simple math expression safely-ish for demo purposes."""
    try:
        # Basic safety: allow only digits, operators, parentheses, whitespace, and decimal point
        import re
        if not re.fullmatch(r"[0-9+\-*/().,\s]+", expr):
            return "error: invalid characters in expression"
        # Replace commas with nothing (in case user writes 1,000)
        expr_clean = expr.replace(",", "")
        # Evaluate
        result = eval(expr_clean)
        return str(result)
    except Exception as e:
        return f"error: {e}"

# Wrap it as a LangChain Tool
calc_tool = Tool(
    name="Calculator",
    func=simple_calc,
    description="Performs basic arithmetic expressions. Example input: '2+3*4' or ' (2+3) * 4 '."
)

tools = [calc_tool]

# Create the agent executor using a ReAct-style agent
agent_executor = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,  # prints the chain of thought / tool calls to stdout
)

# Example usage
print('\nAgent run result:')
print(agent_executor.run("What is 2+3 multiplied by 4?"))

# You can also use the executor to run more complex prompts:
print('\nAnother example:')
print(agent_executor.run("Calculate (2+3)*4 and explain briefly."))

## Capturing intermediate steps / callback handlers

If you want to capture intermediate steps programmatically (rather than printing), use a `CallbackManager` or a custom callback handler. The `verbose=True` flag prints events to stdout, but a callback handler gives you structured access to tool calls, LLM outputs, and more.

Below is a minimal example skeleton (no-op) showing how you'd wire a callback to capture events.

In [ ]:
# Example skeleton for capturing events (adapt per your LangChain version)
from langchain.callbacks.base import BaseCallbackHandler
from langchain.callbacks.manager import CallbackManager

class MyHandler(BaseCallbackHandler):
    def __init__(self):
        self.events = []
    def on_agent_action(self, action, **kwargs):
        # Capture AgentAction events (method name might vary by langchain version)
        self.events.append(("agent_action", action))
    def on_tool_end(self, tool_output, **kwargs):
        self.events.append(("tool_end", tool_output))
    def on_agent_finish(self, finish, **kwargs):
        self.events.append(("agent_finish", finish))

# Instantiate callback manager and re-create the agent with it
handler = MyHandler()
cb_manager = CallbackManager([handler])

# Recreate the agent executor with the callback manager (some versions accept callback_manager=cb_manager)
agent_executor_with_cb = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False,
    # callback_manager=cb_manager  # uncomment if your langchain version supports this parameter
)

# Run (if callback manager is supported in your version you'll capture structured events)
# out = agent_executor_with_cb.run("Who is Sivaprasad valluru?")
# print('Captured events:', handler.events)
print('Note: enable and adapt callback_manager line above if your langchain version supports it.')

## If you truly need a manual agent-action loop

Modern LangChain handles the action/finish loop internally. A manual loop is fragile across versions. If you must, inspect the `AgentExecutor` internals for a `plan` or `a_step` API in your installed version — the exact method names vary.

The recommended approach is to use `initialize_agent` and callbacks as shown above.